### Edmonton Property Geocoding Project

This project converts property addresses from the Edmonton sold listings dataset into geographic coordinates (latitude and longitude) using Python and the geopy library.

The goal is to prepare location-based data for spatial analysis and mapping.

#### Step 1: Import Required Libraries

We import the necessary Python libraries for data processing and geocoding.
- pandas → for data handling
- geopy → for converting addresses into coordinates

In [6]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

#### Step 2: Load the Sold Listings Dataset

We load the sold dataset because it contains actual transaction prices.
This ensures that our analysis is based on real completed sales rather than listing prices.

#####  Why Use Sold Data Instead of Listings

- We are using **sold properties (`sold_df`)** because:
  1. It reflects **actual completed sales**, not just advertised properties.
  2. This ensures our geocoding and analysis are **based on real transactions**, not listings that may never sell.
  3. Helps with **accuracy** for metrics like average price, neighborhood trends, and property location.
  
- Using just **listing data** could be misleading because:
  - Some properties might **stay on the market for months**.
  - Prices listed may **not reflect the final sale price**.
  - Some listings may be **duplicates or removed before sale**.
  
- For our capstone project, sold data gives a **true picture of the real estate market** in Edmonton.

In [10]:
# ------------------------------------------
# Step 2: Load the Dataset
# ------------------------------------------

# Full path to your dataset (adjust if needed)
file_path = r"C:\Users\Adewale Sam\Desktop\DATA3960 - Data Analytics Capstone\RealEstateDataJanuary2026-Data3960-sold.csv"

# Load the CSV file into a pandas DataFrame
import pandas as pd

sold_df = pd.read_csv(file_path, encoding='cp1252', low_memory=False)

# Check the dataset loaded correctly
print("Dataset loaded successfully!")
print("Number of rows and columns:", sold_df.shape)

Dataset loaded successfully!
Number of rows and columns: (515122, 48)


####  Load the Dataset

- We are loading the full real estate dataset for Edmonton using **pandas**.
- The file is a CSV (Comma Separated Values), which is basically a spreadsheet saved as a text file.
- `pd.read_csv()` reads this file into Python so we can work with it.
- `encoding='cp1252'` ensures special characters (like accented letters) are read correctly.
- `low_memory=False` prevents warnings when the dataset has many columns.
- After loading, we print the number of rows and columns to make sure everything is loaded.

### Step 3: Inspect the Dataset

In [11]:
# ------------------------------------------
# Step 3: Inspect the Dataset
# ------------------------------------------

# See the first 5 rows
print(sold_df.head())

# Check column names
print("Columns:", sold_df.columns.tolist())

# Check for missing values
print("Missing values per column:\n", sold_df.isnull().sum())

       Linc # Prop Class Area/City Community            Address Status  \
0    23384076       VLOT      Abee      Abee       48 50 Street      S   
1  0023390842         SF      Abee      Abee     4910 50 STREET      S   
2  0023397698         SF      Abee      Abee     5002 50 Avenue      S   
3    17665648         SF      Acme       NaN   805 CLARK Street      S   
4    17662727         SF      Acme       NaN  738 Clarke Street      S   

   List Price Postal Code   Sold Date  Sold Price  ... Price Per SQFT  \
0       30000     T0A 0A0  2016-08-31       21000  ...           0.00   
1       74900     T0A 0A0  2023-07-27       67000  ...          98.83   
2      150000     T0A 0A0  2024-11-15      137000  ...         104.97   
3       99900     T0M 0A0  2003-06-17       90000  ...          90.00   
4      209900     T0M 0A0  2009-04-22      203000  ...         178.47   

   Sold Pr / List Pr Ratio   Price  Sold Price/Sq Ft  \
0                    70.00   21000              0.00   
1   

#### Inspect the Dataset

- **Why we inspect**: Before doing any analysis, we need to understand the data we have.
- **`.head()`**: Shows the first few rows so we know what the data looks like.
- **`.columns`**: Lists all the columns so we know what info we have (e.g., address, city, price, bedrooms).
- **`.isnull().sum()`**: Checks which columns have missing data. Missing data is common in real-world datasets.
- This helps us **plan data cleaning** and ensures our geocoding works smoothly.

#### Understanding the Dataset

- **Dataset Type:** Real estate transactions (sold properties) for Edmonton.
- **Rows (observations):** Each row represents a **property that was sold**.
- **Columns (features):** 48 columns, including:
    - `Address`, `Community`, `Postal Code` → location of property
    - `Sold Price`, `List Price`, `Price Per SQFT` → price details
    - `Prop Class`, `Style`, `Yr Built` → property characteristics
    - `Buyer Firm`, `Listing Firm` → who sold/bought the property
    - `Lot Sq Metres`, `Garage Y/N`, `Baths`, `Bedrooms` → other property info
- **Missing Values:**
    - Some columns have many missing values, e.g. `Community`, `Postal Code`, `Condo Name`.
    - This is normal in real-world datasets — not every property has every detail.
- **Why this matters:** Knowing where data is missing helps us **clean the data** before analysis or geocoding.
- **Data Quality Note:** Some columns like `Sold Price/Sq Ft` and `List Pr / SqFt` are complete — these are reliable for analysis.

- **Next Step:** Clean the data (Step 4) so that geocoding and analysis can be accurate.

### Step 4: Clean Edmonton Property Data

**Goal:** Prepare the dataset for geocoding by keeping only Edmonton properties and removing rows with missing addresses or communities.

**Why:**  
- We focus only on Edmonton.  
- Missing Address or Community would cause errors during geocoding.  
- Resetting the index keeps the dataset neat and easier to work with.

In [12]:
# Step 4: Clean Edmonton Property Data
import pandas as pd

# Keep only Edmonton properties
edmonton_df = sold_df[sold_df['Area/City'].str.lower() == "edmonton"].copy()

# Remove rows where Address or Community is missing
edmonton_df = edmonton_df.dropna(subset=['Address', 'Community'])

# Reset the row index
edmonton_df.reset_index(drop=True, inplace=True)

print("Step 4 complete. Cleaned Edmonton dataset shape:", edmonton_df.shape)

Step 4 complete. Cleaned Edmonton dataset shape: (344362, 48)


#### Summary: Clean Edmonton Property Data

- We filtered the dataset to **only include Edmonton properties**.  
- Removed rows **without an Address or Community**, because geocoding cannot work with missing locations.  
- Reset the row index to keep the dataset tidy.  

**Result:**  
- Cleaned dataset contains **344,362 rows and 48 columns**.  
- This dataset is now ready for geocoding in the next step.

### Step 5: Geocoding Edmonton Addresses

- Geocoding is the process of converting a **street address** into **latitude and longitude**.  
- We will use the **`geopy` library** with **Nominatim** (OpenStreetMap) as the geocoder.  
- To prevent overload and errors, we **geocode in batches** or **use a sample first** if needed.  
- The resulting dataset will have two new columns:
  - `Latitude`
  - `Longitude`

In [14]:
# Step 5: Fast Geocoding using Google Maps API
import pandas as pd
import requests
import time

# Load your cleaned Edmonton dataset
edmonton_df = sold_df.copy()  # replace with your cleaned dataset

# Your Google Maps API key
API_KEY = "YOUR_GOOGLE_API_KEY"  # <-- replace with your key

# Function to geocode one address using Google Maps API
def geocode_address(address):
    base_url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": address,
        "key": API_KEY
    }
    response = requests.get(base_url, params=params)
    if response.status_code == 200:
        results = response.json().get("results")
        if results:
            location = results[0]["geometry"]["location"]
            return location["lat"], location["lng"]
    return None, None

# Combine address components for better accuracy
edmonton_df['full_address'] = (
    edmonton_df['Address'] + ", " +
    edmonton_df['Community'].fillna('Edmonton') + ", Edmonton, AB, Canada"
)

# Apply geocoding with a short pause to avoid hitting limits
latitudes = []
longitudes = []

for i, addr in enumerate(edmonton_df['full_address']):
    lat, lng = geocode_address(addr)
    latitudes.append(lat)
    longitudes.append(lng)
    
    # Pause every 50 requests to be safe
    if i % 50 == 0:
        time.sleep(0.1)  # adjust if needed

edmonton_df['Latitude'] = latitudes
edmonton_df['Longitude'] = longitudes

# Check results
print(edmonton_df[['full_address', 'Latitude', 'Longitude']].head())

# Optional: Save to CSV
edmonton_df.to_csv("edmonton_geocoded_full.csv", index=False)
print("Geocoding complete. File saved as 'edmonton_geocoded_full.csv'.")

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

####  Geocoding Edmonton Property Addresses

**Goal:** Convert property addresses into **latitude and longitude** so we can plot them on a map or analyze locations.

**What we did to prepare:**
- Identified the key columns needed for geocoding: **Address, Community, Postal Code**.
- Cleaned and organized the dataset so addresses are in a proper format.
- Ensured there are no missing or messy addresses that could cause errors.

**How we geocode:**
1. Use the **`geopy` library** with **Nominatim** (OpenStreetMap).
2. Combine **street address, community, city, province, country** for better accuracy.
3. Apply a **rate limiter** to avoid sending too many requests at once (prevents getting blocked).

**Result:** Two new columns in the dataset:  
- `Latitude` → the north-south coordinate  
- `Longitude` → the east-west coordinate  

**Tips:**  
- For testing, geocode a **sample of 1,000 properties** first.  
- Once the sample works, scale to the **full Edmonton dataset**.

**summary:**  
> Think of this like turning street addresses into GPS coordinates so we can see exactly where each property is on a map. Doing a small test first ensures everything works before geocoding all 344,362 properties.

### Step 6: Sample 1,000 Properties for Geocoding (Testing)

**Goal:** Test geocoding on a smaller dataset first to ensure it works correctly without overloading the service.

**Why we do this:**  
- Geocoding all 344,362 properties at once can take a very long time and may get blocked by the geocoding service.  
- Testing on a **sample of 1,000 properties** allows us to check if the geocoding process runs smoothly and if the coordinates are accurate.

**How we do it:**  
1. Use **`pandas.sample()`** to randomly select 1,000 rows.  
2. Set a **random seed** (`random_state`) so results can be reproduced exactly.  
3. Make a **copy** to avoid accidentally changing the original dataset.  

**Result:**  
- A smaller dataset (`geo_sample`) ready for geocoding.  
- Keeps your full dataset safe while testing.  

**summary:**  
> Imagine you have a huge stack of envelopes to stamp with GPS coordinates. Instead of doing all at once, you take 1,000 envelopes to see if your stamp works. Once it works perfectly, you can stamp the whole stack.

In [ ]:
# Step 6: Sample 1,000 properties for testing
geo_sample = edmonton_df.sample(1000, random_state=42).copy()

print("Sample size for geocoding:", geo_sample.shape)